In [7]:
import pandas as pd
from pathlib import Path
import urllib.request
import zipfile

In [15]:
print("Downloading Brazillian E-Commerce Dataset...")

url = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"
output_path = Path("raw_data")

output_path.mkdir(exist_ok=True)

zip_path = output_path / "dataset.zip"
print("Downloading (2-5 minutes)...")
urllib.request.urlretrieve(url, zip_path)
print("Download Complete!")

print("Extracting files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_path)
print("Extraction Complete!")

print("Verifying files...")
customers = pd.read_csv(output_path / "olist_customers_dataset.csv") 
sellers = pd.read_csv(output_path / "olist_sellers_dataset.csv") 
items = pd.read_csv(output_path / "olist_order_items_dataset.csv") 

print(f"Customers: {len(customers):,}")
print(f"Sellers: {len(sellers):,}")
print(f"Items: {len(items):,}")

print("All Data Downloaded!")



Download Complete!
Extracting files...
Extraction Complete!
Verifying files...
Customers: 99,441
Sellers: 3,095
Items: 112,650
All Data Downloaded!


## Download.py

In [ ]:
# This version improves above script by adding:
# Structured logging (instead of print)
# Error handling (prevents silent failures)
# Validation checks (ensures files exist)
# Cleaner, reusable functions

import subprocess
from pathlib import Path
import zipfile
import pandas as pd
import logging
import sys

# -----------------------------
# Logging Configuration
# -----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -----------------------------
# Define Paths
# -----------------------------
DATA_DIR = Path("data_base\\raw_data")
ZIP_FILE = DATA_DIR / "brazilian-ecommerce.zip"

# -----------------------------
# Step 1: Create Directory
# -----------------------------
def create_data_dir():
    if not DATA_DIR.exists():
        DATA_DIR.mkdir()
        logging.info("Data directory created.")
    else:
        logging.info("Data directory already exists.")

# -----------------------------
# Step 2: Download Dataset
# -----------------------------
def download_dataset():
    logging.info("Starting dataset download...")

    try:
        subprocess.run([
            "kaggle", "datasets", "download",
            "-d", "olistbr/brazilian-ecommerce",
            "-p", str(DATA_DIR)
        ], check=True)

        logging.info("Download completed successfully.")

    except subprocess.CalledProcessError:
        logging.error("Download failed. Check Kaggle API setup.")
        sys.exit(1)

# -----------------------------
# Step 3: Extract Dataset
# -----------------------------
def extract_dataset():
    logging.info("Extracting dataset...")

    if not ZIP_FILE.exists():
        logging.error("Zip file not found. Extraction aborted.")
        sys.exit(1)

    try:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)

        logging.info("Extraction completed.")

    except zipfile.BadZipFile:
        logging.error("Invalid zip file.")
        sys.exit(1)

# -----------------------------
# Step 4: Validate Data
# -----------------------------
def validate_data():
    logging.info("Validating extracted files...")

    try:
        customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
        sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
        items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

        logging.info(f"Customers: {len(customers):,}")
        logging.info(f"Sellers: {len(sellers):,}")
        logging.info(f"Items: {len(items):,}")

    except FileNotFoundError as e:
        logging.error(f"Missing file: {e}")
        sys.exit(1)

    except Exception as e:
        logging.error(f"Unexpected error: {e}")
        sys.exit(1)

# -----------------------------
# Pipeline Orchestration
# -----------------------------
def run_pipeline():
    logging.info("ETL Pipeline Started")

    create_data_dir()
    download_dataset()
    extract_dataset()
    validate_data()

    logging.info("ETL Pipeline Completed Successfully")

# -----------------------------
# Entry Point
# -----------------------------
if __name__ == "__main__":
    run_pipeline()

## Explore.py

In [ ]:
import pandas as pd
from pathlib import Path

def explore_data():
    data_path = Path("raw")
    files = [
        data_path / "olist_customers_dataset.csv",
        data_path / "olist_sellers_dataset.csv",
        data_path / "olist_order_items_dataset.csv"
    ]
    
    print("Data is available... exploring now...")
    for file in files:
        
        df = pd.read_csv(data_path / file)
        print(f"Shape of {file.name}: {df.shape}")
        print(f"Columns in {file.name}: {df.columns.tolist()}")
        print(f"Data types in {file.name}: {df.dtypes.tolist()}")
        print(f"First few rows of {file.name}:\n{df.head()}")
        
explore_data()


## Transform.py

In [ ]:
import pandas as pd
import logging
from pathlib import Path


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Define the path to the dataset
RAW_DATA_DIR = Path("data_base\\raw_data")
PROCESSED_DATA_DIR = Path("data_base\\processed_data")


# Create processed data directory if it doesn't exist
def create_processed_data_dir():
    if not PROCESSED_DATA_DIR.exists():
        PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
        logging.info("Processed data directory created.")
    else:
        logging.info("Processed data directory already exists.")


def load_and_transform_data():
    logging.info("Loading and transforming data...")

    try:
        create_processed_data_dir()

        # Load datasets
        logging.info("Loading data...")
        customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
        sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
        items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
        orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
        payments = pd.read_csv(
            RAW_DATA_DIR / "olist_order_payments_dataset.csv")

        # Clean datasets
        logging.info("Cleaning customer dataset...")
        clean_customers = customers.drop_duplicates(
            subset="customer_id").copy()
        clean_customers["country"] = "Brazil"

        logging.info("Cleaning sellers dataset...")
        clean_sellers = sellers.drop_duplicates(subset="seller_id").copy()

        logging.info(
            "Cleaning Payment Dataset and Merging orders, items, and payments datasets...")
        payments = payments.drop_duplicates(subset="order_id").copy()
        transactions = (
            orders
            .merge(items, on="order_id", how="left")
            .merge(payments, on="order_id", how="left")
        )

        logging.info("Converting date columns to datetime format...")
        date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                     "order_delivered_customer_date", "order_estimated_delivery_date"]
        for col in date_cols:
            transactions[col] = pd.to_datetime(
                transactions[col], errors='coerce')

        # Save cleaned datasets
        logging.info("Saving cleaned datasets...")
        clean_customers.to_csv(PROCESSED_DATA_DIR /
                               "clean_customers.csv", index=False)
        clean_sellers.to_csv(PROCESSED_DATA_DIR /
                             "clean_sellers.csv", index=False)
        transactions.to_csv(PROCESSED_DATA_DIR /
                            "transactions.csv", index=False)

        logging.info(f"Customers rows: {len(clean_customers):,}")
        logging.info(f"Sellers rows: {len(clean_sellers):,}")
        logging.info(f"Transactions rows: {len(transactions):,}")
        logging.info("Data transformation completed successfully.")

    except Exception as e:
        logging.error(f"Error during data transformation: {e}")


if __name__ == "__main__":
    load_and_transform_data()


## More Cleaner Version of Transform.py


In [ ]:
import pandas as pd
import logging
from pathlib import Path
import traceback

# ----------------------------
# Logging configuration
# ----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ----------------------------
# Paths
# ----------------------------
RAW_DATA_DIR = Path("data_base") / "raw_data"
PROCESSED_DATA_DIR = Path("data_base") / "processed_data"
OUTPUT_FORMAT = "csv"  

# ----------------------------
# Helper Functions
# ----------------------------
def create_dir(path: Path):
    """Create directory if it does not exist."""
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)
        logging.info(f"Directory created: {path.resolve()}")
    else:
        logging.info(f"Directory already exists: {path.resolve()}")


def load_csv(file_path: Path) -> pd.DataFrame:
    """Load CSV and log shape."""
    if not file_path.exists():
        logging.error(f"File not found: {file_path}")
        raise FileNotFoundError(f"File not found: {file_path}")
    df = pd.read_csv(file_path)
    logging.info(f"Loaded {file_path.name} with {df.shape[0]:,} rows and {df.shape[1]} columns")
    return df


def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    """Clean customer dataset."""
    df_clean = df.drop_duplicates(subset="customer_id").copy()
    df_clean["country"] = "Brazil"
    logging.info(f"Cleaned customers: {df_clean.shape[0]:,} rows")
    return df_clean


def clean_sellers(df: pd.DataFrame) -> pd.DataFrame:
    """Clean sellers dataset."""
    df_clean = df.drop_duplicates(subset="seller_id").copy()
    logging.info(f"Cleaned sellers: {df_clean.shape[0]:,} rows")
    return df_clean


def merge_transactions(orders: pd.DataFrame, items: pd.DataFrame, payments: pd.DataFrame) -> pd.DataFrame:
    """Merge orders, items, and payments into transactions."""
    payments_clean = payments.drop_duplicates(subset="order_id").copy()
    transactions = (
        orders
        .merge(items, on="order_id", how="left", validate="one_to_many")
        .merge(payments_clean, on="order_id", how="left", validate="one_to_one")
    )
    logging.info(f"Merged transactions: {transactions.shape[0]:,} rows")
    return transactions


def convert_date_columns(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Convert columns to datetime, ignoring missing columns."""
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            logging.info(f"Converted column to datetime: {col}")
        else:
            logging.warning(f"Column not found, skipping datetime conversion: {col}")
    return df


def save_dataframe(df: pd.DataFrame, path: Path, format: str = "csv"):
    """Save DataFrame to CSV or Parquet."""
    if format == "csv":
        df.to_csv(path.with_suffix(".csv"), index=False)
        logging.info(f"Saved CSV: {path.with_suffix('.csv')}")
    elif format == "parquet":
        df.to_parquet(path.with_suffix(".parquet"), index=False)
        logging.info(f"Saved Parquet: {path.with_suffix('.parquet')}")
    else:
        logging.error(f"Unsupported format: {format}")
        raise ValueError(f"Unsupported format: {format}")


# ----------------------------
# ETL Function
# ----------------------------
def run_etl():
    try:
        logging.info("Starting ETL pipeline...")

        # Ensure output directory exists
        create_dir(PROCESSED_DATA_DIR)

        # Load raw datasets
        customers = load_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
        sellers = load_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
        items = load_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
        orders = load_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
        payments = load_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")

        # Clean datasets
        clean_cust = clean_customers(customers)
        clean_sell = clean_sellers(sellers)

        # Merge transactions
        transactions = merge_transactions(orders, items, payments)

        # Convert date columns
        date_columns = [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
        transactions = convert_date_columns(transactions, date_columns)

        # Save cleaned datasets
        save_dataframe(clean_cust, PROCESSED_DATA_DIR / "clean_customers", OUTPUT_FORMAT)
        save_dataframe(clean_sell, PROCESSED_DATA_DIR / "clean_sellers", OUTPUT_FORMAT)
        save_dataframe(transactions, PROCESSED_DATA_DIR / "transactions", OUTPUT_FORMAT)

        logging.info("ETL pipeline completed successfully!")

    except Exception as e:
        logging.error(f"ETL pipeline failed: {e}")
        logging.error(traceback.format_exc())


# ----------------------------
# Entry Point
# ----------------------------
if __name__ == "__main__":
    run_etl()